# Guide01: Defining the Problem, Understanding, Cleaning, Splitting, and Exploring the Data

This is the first of six notebooks that work through one real project end to end: predicting house sale prices in Ames, Iowa. Together they follow the [13-stage workflow](../Guide00_Supervised-ML_Linear_Regression_end-to-end_workflow.md) from `Guide00`. This notebook covers **Stages 1-5**: define the problem, understand the data, clean it, split it, and explore the training set. By the end, we will have a locked test set we do not touch again until `Guide06`, and a written to-do list for `Guide02`.

**Prerequisite:** [Guide00: An End-to-End Linear Regression Workflow](../Guide00_Supervised-ML_Linear_Regression_end-to-end_workflow.md).


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 15)
plt.rcParams["figure.figsize"] = (7, 4)


---
## Stage 1. Define the Problem

**Question:** how much will a house in Ames, Iowa sell for, given its characteristics?

* **Target ($y$):** `SalePrice`, in dollars.
* **Features ($X$):** the house's physical characteristics, location, and condition at time of sale (everything else in the table).
* **Success metric:** MAE (mean absolute error), in dollars, so the score is easy to explain ("typically off by about \$X"). We will also track RMSE and $R^2$ from `Guide03` onward, but MAE is the number we optimize for and report first.
* **Objective:** mainly *prediction* (how accurate is the estimate?), with *interpretation* (which features drive price, and by how much?) as a secondary goal we return to in `Guide03` and `Guide06`.
* **Good enough:** clearly better than guessing the average sale price for every house — that comparison is our reference baseline, built in `Guide03`.
* **Use case:** a rough pricing aid for a single market (Ames), not a general-purpose home-price model.


---
## Stage 2. Understand the Data

* **Source:** individual residential property sales in Ames, Iowa, assessed by the city and compiled for a well-known teaching dataset (the same lineage as the Kaggle "House Prices" competition).
* **One row:** one arms-length sale of one house.
* **Time span:** sales from 2006 through 2010 — which includes the 2008 housing crash. A model trained here reflects *that* market, in *that* city; it says little about Boston or about 2024 prices.
* **The target:** `SalePrice`, the price the house actually sold for.

Before touching the data, it is worth reading the data dictionary for traps. Three are easy to miss:


In [ ]:
from pipeline.ames_workflow import load_ames

raw = load_ames()
print(raw.shape)
raw[["MSSubClass", "OverallQual", "ExterQual", "MoSold", "YrSold"]].head()


| Column | Looks like | Actually is |
| :--- | :--- | :--- |
| `MSSubClass` | A number (20, 60, 120, ...) | A building-type *category* — a numeric code, not a quantity. Averaging it would be meaningless. |
| `OverallQual`, `OverallCond` | Numbers 1-10 | Genuinely ordinal (higher is better), safe to treat as numeric. |
| `ExterQual`, `KitchenQual`, and similar `*Qual`/`*Cond` columns | Text | Ordinal quality grades, `Po < Fa < TA < Gd < Ex`. Not yet numbers, but not unordered categories either. |
| `MoSold` | A number 1-12 | The calendar month — a category (December is not "twelve times" January), even though it is stored as an integer. |

`MSSubClass` gets fixed in Stage 3 (it is a **wrong data type**, in `Guide00`'s sense). The quality grades and `MoSold` are noted here and handled when we encode, in `Guide02`.


### Missing, or just a different category?

`pandas.read_csv` treats the literal text `"None"` as one of its default missing-value markers. That is a problem here, because several columns in Ames genuinely use `"None"` as a category label — "no pool", "no alley access", "no basement" — not as a missing value. Reading the file the normal way hides this:


In [ ]:
raw_default = pd.read_csv("data/Ames_Housing_Sales.csv")  # pandas' default NA parsing
missing = raw_default.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"columns with 'missing' values under default parsing: {len(missing)}")
print(f"total 'missing' cells: {missing.sum():,}")
missing.head()


Every one of those columns is a **structural "None"**, not a gap in the data. Look at `PoolQC` (pool quality) up close:


In [ ]:
print("Default parsing (None -> NaN):")
print(raw_default["PoolQC"].value_counts(dropna=False))
print()
print("keep_default_na=False (None kept as a real category):")
print(raw["PoolQC"].value_counts())


Only 7 of 1,379 houses have a pool at all. The other 1,372 do not — `"None"` is the honest, correct label for "no pool," and it is by far the most common value.

**Why this matters:** imagine cleaning this data *without* first checking what "missing" means here. A common reflex is to fill missing categorical values with the most frequent one. Under the *default* parsing, the most frequent **non-null** value of `PoolQC` is `"Gd"` (good) — so a naive imputer would tell 1,372 pool-less houses that they have a *good pool*. That is exactly backward, and it is why Stage 2 (understand) comes before Stage 3 (clean): you cannot safely fix what you have not first understood.

This is why `load_ames()` in `pipeline/ames_workflow.py` reads the file with `keep_default_na=False`.


---
## Stage 3. Clean the Data

Two fixes, both safe to do *before* the split because neither one learns anything from the data — they apply the same way regardless of which rows end up in training or test:

1. **Fix the wrong data type.** Cast `MSSubClass` to text, so it is treated as a category from `Guide02` onward instead of being averaged like a real number.
2. **Remove two documented anomalies.** Four houses have `GrLivArea` (above-ground living area) over 4,000 square feet. Two of them are unusually cheap for their size:


In [ ]:
big = raw.loc[raw["GrLivArea"].astype(float) > 4000,
              ["GrLivArea", "SalePrice", "SaleCondition", "YrSold"]]
big


The two `Partial`-condition sales (rows 493 and 1226) were **new construction sold before it was finished** — their low price reflects an unfinished house, not a fair market value for that much space. That is a real reason to remove them: keeping them would teach the model that huge houses can be cheap, which is not true of a normal, finished sale. The other two large houses are `Normal`/`Abnorml` sales and stay — being large is not, by itself, a reason to drop a row.

We do **not** touch missing values here. Every "missing" cell in this file turned out to be the category `"None"` (Stage 2), so there is nothing to impute. `Guide02` practices real imputation on a labelled simulation instead.


In [ ]:
from pipeline.ames_workflow import clean_ames

print("duplicate rows:", raw.duplicated().sum())  # a routine check, even though the answer is 0 here

clean = clean_ames(raw)
print(f"rows: {len(raw)} -> {len(clean)}  ({len(raw) - len(clean)} removed)")
print("MSSubClass dtype:", clean["MSSubClass"].dtype)


---
## Stage 4. Split Into Training and Test Sets

`Guide00` recommends a random split by default. Here we use a **time-based** split instead: train on sales through 2009, and hold out **2010 alone** as the test set. A random split would mix 2010 sales into training, which is not how this model will actually be used — it will always be asked to price a sale that has not happened yet. Splitting by year matches that, and it means the single Stage 10 look at the test set (`Guide06`) doubles as a check for drift (Stage 12) against a genuinely future year, instead of needing a second, separate demonstration.

**From here on, the 2010 rows are off limits** — no plots, no statistics, no tuning — until `Guide06`.


In [ ]:
from pipeline.ames_workflow import split_ames

X_train, X_test, y_train, y_test = split_ames(clean)
print(f"train (2006-2009): {X_train.shape}   test (2010, locked): {X_test.shape}")
print(f"test share: {len(X_test) / len(clean):.1%}")


---
## Stage 5. Explore the Data (Training Set Only)

Everything from here on uses `X_train` / `y_train` only.

### The target


In [ ]:
from scipy.stats import skew

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y_train, bins=40)
axes[0].set(title="SalePrice", xlabel="dollars")
axes[1].hist(np.log(y_train), bins=40)
axes[1].set(title="log(SalePrice)", xlabel="log-dollars")
plt.tight_layout()
plt.show()

print(f"SalePrice      skew: {skew(y_train):.2f}")
print(f"log(SalePrice) skew: {skew(np.log(y_train)):.2f}")


`SalePrice` has a long right tail (a handful of expensive houses) — a skew of about 1.9. On the log scale that drops to about 0.3, close to symmetric. That is the signal `Guide02` will act on: transform the target before fitting.

### Relationships with the target


In [ ]:
numeric_train = X_train.select_dtypes("number").copy()
numeric_train["SalePrice"] = y_train
correlations = numeric_train.corr()["SalePrice"].drop("SalePrice").sort_values(ascending=False)
correlations.head(8)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_train["GrLivArea"], y_train, s=8, alpha=0.4)
axes[0].set(title="Living area vs. price", xlabel="GrLivArea (sq ft)", ylabel="SalePrice")
axes[1].boxplot(
    [y_train[X_train["OverallQual"] == q] for q in sorted(X_train["OverallQual"].unique())],
    tick_labels=sorted(X_train["OverallQual"].unique()),
)
axes[1].set(title="Price by overall quality", xlabel="OverallQual", ylabel="SalePrice")
plt.tight_layout()
plt.show()


`OverallQual` (a 1-10 quality rating) and `GrLivArea` (living area) are the two strongest numeric relationships with price. Price rises with quality in fairly even steps — encouraging for a *linear* model — and grows with living area, though the spread widens for larger, pricier houses (worth remembering when we look at residuals in `Guide03`).

### Correlated features (a preview of Stage 9)


In [ ]:
num_features = numeric_train.drop(columns="SalePrice")
corr_matrix = num_features.corr().abs()
pairs = [
    (a, b, round(corr_matrix.loc[a, b], 2))
    for i, a in enumerate(corr_matrix.columns)
    for b in corr_matrix.columns[i + 1:]
    if corr_matrix.loc[a, b] > 0.8
]
pairs


A handful of feature pairs are strongly correlated with *each other* — for example the number of garage cars and the garage's square footage. That is **multicollinearity**, and it is exactly what regularization (`Guide05`) is for; we note it now and act on it later.

### Rare categories


In [ ]:
X_train["Neighborhood"].value_counts().tail(5)


A few neighborhoods have only a handful of sales in the training data. One-hot encoding every neighborhood as its own column (`Guide02`) would give some of these categories almost no data to learn from — a reason to consider grouping rare levels later.


---
## Stage 6 To-Do List

Exploration points to a clear set of jobs for `Guide02`:

* **Transform the target.** `SalePrice` is skewed; fit the model on `log(SalePrice)` and remember to convert predictions back.
* **Encode about 43 categorical columns**, including the ordinal quality grades and `MSSubClass`/`MoSold`.
* **Group rare categories** (some `Neighborhood` levels have very few sales) before one-hot encoding.
* **Expect collinearity** among the garage and living-area features; plan for regularization later.
* **No real missing values here** — but `Guide02` will still build and test an imputer, on a labelled simulation, since most real datasets do have genuine gaps.

## What We Hand to `Guide02`

* `pipeline/ames_workflow.py`, with `load_ames()`, `clean_ames()`, and `split_ames()`.
* The locked test set: 164 sales from 2010. Not to be opened again until `Guide06`.
* The to-do list above.


---
## Your Turn

Pick **one** of these and repeat Stages 2-4 on it (understand, clean, split). Both datasets live in `data/`.

**Option A: `CarPrice_Assignment.csv`.** The `CarName` column mixes the brand into the full car name, and several brand names are misspelled or inconsistently cased (`maxda` for Mazda, `porcshce` and `Nissan` vs. `nissan`, `toyouta`, `vokswagen`/`vw` for Volkswagen). Extract the brand, fix the typos, and check: how many distinct "brands" did you start with, and how many after cleaning?

**Option B: `california_housing_price.csv`.** The target (median house value) is capped: every house worth $500,001 or more is recorded as exactly the cap value. Find how many rows sit exactly at the cap, and write a sentence on what that does to a model trained on this data — would MAE for expensive houses be trustworthy?


In [ ]:
# Your turn: load the dataset you picked and start with `df.info()` and a data dictionary read-through.
